In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 04. Clasificación, recurrentes, atípicos, sentimiento y recomendador

Cada técnica frente a su alternativa sencilla.

In [2]:
C = M['clasificacion']; print(json.dumps(C['glosa_a_categoria'], indent=1)); print(json.dumps(C['recurrentes'], indent=1)); print(json.dumps(C['suscripciones_duplicadas'], indent=1))

{
 "diccionario": {
  "f1_macro": 0.8642178793235947,
  "exactitud": 0.9894333333333334
 },
 "tfidf_char_sgd_lineal": {
  "f1_macro": 0.8936542149159054,
  "exactitud": 0.9894666666666667
 },
 "cobertura_diccionario": 0.9999666666666667,
 "n_prueba": 30000
}
{
 "periodicidad_y_monto": {
  "precision": 0.9985535607929891,
  "recall": 0.8752330524274741,
  "f1": 0.9328352277243462
 },
 "baseline_mismo_comercio_3_veces": {
  "precision": 0.09207255705549877,
  "recall": 1.0,
  "f1": 0.1686198530686449
 },
 "n_clientes": 150
}
{
 "precision": 1.0,
 "recall": 0.7,
 "f1": 0.8235294117647058,
 "detectados": 7,
 "reales": 10
}


In [3]:
print(json.dumps(C['atipicos'], indent=1)); print(json.dumps(C['sentimiento'], indent=1))

{
 "n_atipicos": 2066,
 "n": 800000,
 "zscore_robusto": {
  "precision_en_k": 0.6878025169409487,
  "recall_en_k": 0.6878025169409487
 },
 "isolation_forest": {
  "precision_en_k": 0.750242013552759,
  "recall_en_k": 0.750242013552759
 }
}
{
 "plantillas_futuras": {
  "lexico_f1": 0.6184785043128144,
  "tfidf_lr_f1": 1.0,
  "n": 12801
 },
 "escritas_a_mano": {
  "lexico_f1": 0.9163752913752914,
  "tfidf_lr_f1": 0.7306915306915307,
  "n": 24
 }
}


## Impacto en soles de seguir a Lucas (reproducción contrafactual 2024–2025)

In [4]:
I = M['impacto']; display(tabla(I['tabla'])); print(json.dumps(I['diferencias'], indent=1)); display(tabla(I['por_segmento'])); I['supuestos']

,dias_en_rojo_por_cliente_mes,colchon_final_soles,colchon_final_meses_de_ingreso,costo_de_cubrir_el_hueco_soles_por_mes
nada,0.837,4263.586,1.867,1.486
diez_fijo,13.365,6188.377,2.909,78.816
diez_fijo_ajustado,0.837,10411.295,4.757,1.486
lucas,0.594,4585.284,2.017,1.095


{
 "diez_fijo_vs_nada": {
  "dias_rojo_evitados_por_cliente_mes": {
   "media": -12.52781040063902,
   "ic95": [
    -12.884035046000534,
    -12.177838645488858
   ]
  },
  "colchon_adicional_soles": {
   "media": 1924.7913593127812,
   "ic95": [
    1767.994949606708,
    2077.0261151970167
   ]
  },
  "intereses_evitados_soles_por_mes": {
   "media": -77.33001388015421,
   "ic95": [
    -82.46885471004205,
    -72.19206394534484
   ]
  }
 },
 "diez_fijo_vs_diez_fijo_ajustado": {
  "dias_rojo_evitados_por_cliente_mes": {
   "media": -12.52781040063902,
   "ic95": [
    -12.899584567101101,
    -12.152690417722516
   ]
  },
  "colchon_adicional_soles": {
   "media": -4222.917674621881,
   "ic95": [
    -4460.553804099789,
    -3957.1669336879636
   ]
  },
  "intereses_evitados_soles_por_mes": {
   "media": -77.33001388015421,
   "ic95": [
    -83.21400940133991,
    -71.44431014554763
   ]
  }
 },
 "diez_fijo_ajustado_vs_nada": {
  "dias_rojo_evitados_por_cliente_mes": {
   "media": 0

,dias_rojo_nada,dias_rojo_lucas,colchon_nada,colchon_lucas
base_comision,0.266,0.155,5013.120,5347.711
formal_quincena,0.297,0.163,6656.588,7029.449
independiente_diario,0.785,0.543,2365.977,2523.740
independiente_proyectos,1.879,1.528,3229.730,3616.599
mixto,1.331,0.863,2763.352,3135.119


{'tasa_mensual_por_cubrir_el_hueco': 0.1,
 'fraccion_apartada_10_fijo': 0.1,
 'recorte_hormiga': 0.2,
 'recorte_hormiga_en_alerta': 0.5,
 'dias_de_alerta': 14,
 'umbral_alerta_p_rojo': 0.25,
 'asignacion_sueldo_lucas': 0.95,
 'umbral_irregular_cv': 0.4}

## Recomendaciones y ofertas para tres perfiles

In [5]:
os.environ['LLM_PROVIDER'] = 'plantillas'
from src.agente_analitico.repositorio import Repositorio
repo = Repositorio()
sys.path.insert(0, os.path.join(RAIZ, 'eval'))
from evaluar_agente import clientes_de_prueba
ids = clientes_de_prueba(repo)
from src.models import recomendador as R
for nombre, cid in ids.items():
    c = repo.contexto(cid); print(nombre, cid, c['nivel'], 'salud', c['salud'])
    for r in R.recomendar(c, 4): print('  -', r['id'], '|', r['accion'][:110])

independiente_en_zona_del_nino C00045 maestro salud 72.1
  - prevencion_fen | Vives en Piura, zona con exposicion alta a lluvias por El Nino: prepara tu hogar y un fondo de S/ 7,144.
  - ahorro_apartados | Abre un apartado para tu colchon y aporta de a poco: tu meta es llegar a 4.5 meses de ingreso.
  - sueldo_lucas | Activa tu Sueldo Lucas: guarda lo que te sobre en meses buenos y usa un monto fijo de unos S/ 2,491 por quince
  - ahorro_redondeo | Activa el ahorro por redondeo: cada gasto se sube al sol siguiente y la diferencia va a tu apartado, unos S/ 2
dependiente_en_lima C00003 campeon salud 84.7
  - ahorro_apartados | Abre un apartado para tu colchon y aporta de a poco: tu meta es llegar a 3.0 meses de ingreso.
  - prevencion_fen | Vives en Lima, zona con exposicion media a lluvias por El Nino: prepara tu hogar y un fondo de S/ 2,038.
  - ahorro_redondeo | Activa el ahorro por redondeo: cada gasto se sube al sol siguiente y la diferencia va a tu apartado, unos S/ 3
  - ordenar_d